In [ ]:
import math
from datetime import datetime, timedelta

# ===================== 核心函数：太阳位置计算（保留，逻辑正确） =====================
def calculate_sun_position(date_time: datetime, lat: float, lng: float) -> dict:
    PI = math.pi
    RAD = PI / 180.0
    DAY_MS = 1000 * 60 * 60 * 24
    J1970 = 2440588.0
    J2000 = 2451545.0
    EARTH_OBLIQUITY = 23.4397 * RAD

    def to_julian(dt: datetime) -> float:
        timestamp_ms = dt.timestamp() * 1000
        return timestamp_ms / DAY_MS - 0.5 + J1970

    def to_days(dt: datetime) -> float:
        return to_julian(dt) - J2000

    def right_ascension(l: float, b: float) -> float:
        return math.atan2(math.sin(l) * math.cos(EARTH_OBLIQUITY) - math.tan(b) * math.sin(EARTH_OBLIQUITY),
                          math.cos(l))

    def declination(l: float, b: float) -> float:
        return math.asin(math.sin(b) * math.cos(EARTH_OBLIQUITY) +
                         math.cos(b) * math.sin(EARTH_OBLIQUITY) * math.sin(l))

    def azimuth(H: float, phi: float, dec: float) -> float:
        return math.atan2(math.sin(H),
                          math.cos(H) * math.sin(phi) - math.tan(dec) * math.cos(phi))

    def altitude(H: float, phi: float, dec: float) -> float:
        return math.asin(math.sin(phi) * math.sin(dec) +
                         math.cos(phi) * math.cos(dec) * math.cos(H))

    def sidereal_time(d: float, lw: float) -> float:
        return RAD * (280.16 + 360.9856235 * d) - lw

    def solar_mean_anomaly(d: float) -> float:
        return RAD * (357.5291 + 0.98560028 * d)

    def ecliptic_longitude(M: float) -> float:
        C = RAD * (1.9148 * math.sin(M) + 0.02 * math.sin(2 * M) + 0.0003 * math.sin(3 * M))
        P = RAD * 102.9372
        return M + C + P + PI

    def sun_coords(d: float) -> dict:
        M = solar_mean_anomaly(d)
        L = ecliptic_longitude(M)
        return {'dec': declination(L, 0.0), 'ra': right_ascension(L, 0.0)}

    lw = RAD * -lng
    phi = RAD * lat
    d = to_days(date_time)
    sun_c = sun_coords(d)
    H = sidereal_time(d, lw) - sun_c['ra']
    az_rad = azimuth(H, phi, sun_c['dec'])
    alt_rad = altitude(H, phi, sun_c['dec'])

    az_deg = (math.degrees(az_rad) + 180) % 360  # 修正为正北0°顺时针
    alt_deg = math.degrees(alt_rad)
    return {'azimuth': az_deg, 'altitude': alt_deg}

# ===================== 新增函数：计算太阳方向向量（东北天ENU坐标系） =====================
def calculate_sun_direction_vector(date_time: datetime, lat: float, lng: float) -> dict:
    """
    计算地面观察者视角下的太阳单位方向向量（东北天ENU坐标系）
    返回：包含x(东)、y(北)、z(天顶)分量和单位向量的字典
    """
    # 1. 获取太阳的方位角和仰角
    sun_pos = calculate_sun_position(date_time, lat, lng)
    az_deg = sun_pos['azimuth']
    alt_deg = sun_pos['altitude']

    # 2. 转换为弧度制（数学计算必需）
    az_rad = math.radians(az_deg)
    alt_rad = math.radians(alt_deg)

    # 3. 按照推导公式计算三个分量
    cos_alt = math.cos(alt_rad)  # 提取公共项，简化计算
    x_east = cos_alt * math.sin(az_rad)   # 东向（x轴）分量
    y_north = cos_alt * math.cos(az_rad)  # 北向（y轴）分量
    z_up = math.sin(alt_rad)              # 天顶（z轴）分量

    # 4. 验证单位向量（模长应接近1，因浮点误差可能略有偏差）
    vector_magnitude = math.sqrt(x_east**2 + y_north**2 + z_up**2)

    # 5. 整理返回结果
    return {
        'azimuth_deg': az_deg,          # 方位角（°）
        'altitude_deg': alt_deg,        # 仰角（°）
        'x_east': x_east,               # 东向分量
        'y_north': y_north,             # 北向分量
        'z_up': z_up,                   # 天顶分量
        'unit_vector': (x_east, y_north, z_up),  # 单位方向向量
        'magnitude': vector_magnitude   # 向量模长（验证用，应≈1）
    }


In [ ]:
# ===================== 新增：分解太阳光照在东南西墙面的垂直强度分量 =====================
def calculate_wall_sunlight_intensity(date_time: datetime, lat: float, lng: float) -> dict:
    """
    计算北半球楼房东、南、西三个垂直墙面的太阳垂直照射强度分量（以单位太阳光照为基准）
    返回：各墙面的垂直照射强度（0 表示无垂直光照，>0 表示有垂直光照，值越大强度越高）
    """
    # 1. 复用之前的函数，获取太阳方向向量
    sun_dir = calculate_sun_direction_vector(date_time, lat, lng)
    x_east = sun_dir['x_east']  # 东向分量（水平）
    y_north = sun_dir['y_north']  # 北向分量（水平）
    alt_deg = sun_dir['altitude_deg']  # 仰角（仅保留，用于判断太阳是否在地平线上）

    # 2. 先判断太阳是否在地平线上（alt > 0 才有有效光照）
    if alt_deg <= 0:
        return {
            'azimuth_deg': sun_dir['azimuth_deg'],
            'altitude_deg': alt_deg,
            'east_wall_intensity': 0.0,
            'south_wall_intensity': 0.0,
            'west_wall_intensity': 0.0,
            'note': '太阳在地平线以下，无有效光照'
        }

    # 3. 按照推导公式，计算三个墙面的垂直照射强度分量（取 max(0, 点积)，无负光照）
    east_intensity = max(0.0, x_east)  # 东面墙：仅东向分量为正时有光照
    south_intensity = max(0.0, -y_north)  # 南面墙：仅北向分量为负（即正南方向）时有光照
    west_intensity = max(0.0, -x_east)  # 西面墙：仅东向分量为负（即正西方向）时有光照

    # 4. 整理返回结果
    return {
        'azimuth_deg': sun_dir['azimuth_deg'],
        'altitude_deg': alt_deg,
        'east_wall_intensity': east_intensity,
        'south_wall_intensity': south_intensity,
        'west_wall_intensity': west_intensity,
        'note': '太阳在地平线以上，光照分量有效（以单位太阳光照为基准）'
    }


from datetime import datetime, timedelta
# 追加可视化所需导入
import matplotlib.pyplot as plt
import numpy as np

# ===================== 新增：可视化一天内三面墙的垂直日照强度趋势 =====================
def plot_daily_wall_sunlight_trend(target_date: datetime, lat: float, lng: float, time_interval_min=10):
    """
    可视化指定日期、指定地点，一天内东、南、西三面墙的垂直日照强度趋势
    :param target_date: 目标日期（datetime，只需年月日，时分秒会被忽略）
    :param lat: 纬度
    :param lng: 经度
    :param time_interval_min: 采样时间间隔（分钟），默认10分钟，间隔越小曲线越平滑
    """
    # 1. 配置matplotlib中文显示（避免乱码）
    plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei']
    plt.rcParams['axes.unicode_minus'] = False
    plt.rcParams['figure.figsize'] = (12, 6)
    plt.rcParams['grid.alpha'] = 0.3

    # 2. 构造当天的采样时间序列（从00:00到23:59，按指定间隔采样）
    target_date = datetime(target_date.year, target_date.month, target_date.day)  # 忽略时分秒，只保留年月日
    sample_times = [
        target_date + timedelta(minutes=i)
        for i in range(0, 24 * 60, time_interval_min)
    ]

    # 3. 初始化存储结果的列表
    valid_times = []  # 有效时间（太阳在地平线上，格式为字符串，用于横轴）
    east_intensities = []  # 东墙强度
    south_intensities = []  # 南墙强度
    west_intensities = []  # 西墙强度

    # 4. 遍历采样时间，计算各墙面光照强度
    for sample_time in sample_times:
        wall_light = calculate_wall_sunlight_intensity(sample_time, lat, lng)
        alt_deg = wall_light['altitude_deg']

        # 仅保留太阳在地平线上的有效数据（过滤夜间/日出日落前后无效数据）
        if alt_deg > 0:
            valid_times.append(sample_time.strftime("%H:%M"))
            east_intensities.append(wall_light['east_wall_intensity'])
            south_intensities.append(wall_light['south_wall_intensity'])
            west_intensities.append(wall_light['west_wall_intensity'])

    # 5. 绘制可视化图表
    fig, ax = plt.subplots()

    # 绘制三条趋势线，区分颜色和标签
    ax.plot(valid_times, east_intensities, color='#1E90FF', label='东墙', linewidth=2.5, marker='.', markersize=3)
    ax.plot(valid_times, south_intensities, color='#FF4500', label='南墙', linewidth=2.5, marker='.', markersize=3)
    ax.plot(valid_times, west_intensities, color='#32CD32', label='西墙', linewidth=2.5, marker='.', markersize=3)

    # 6. 图表美化与标注
    ax.set_title(f'一天内东/南/西墙垂直日照强度趋势\n日期：{target_date.strftime("%Y-%m-%d")} | 纬度：{lat}° | 经度：{lng}°',
                 fontsize=14, pad=20)
    ax.set_xlabel('当天时间', fontsize=12)
    ax.set_ylabel('垂直日照强度（单位太阳光照基准）', fontsize=12)
    ax.set_ylim(0, 1.1)  # 强度最大值为1，预留少量空间更美观
    ax.grid(True, linestyle='--')
    ax.legend(fontsize=10)

    # 优化横轴时间标签（避免过于密集，每2小时显示一个标签）
    step = max(1, len(valid_times) // 12)  # 控制标签密度，适配不同采样间隔
    ax.set_xticks(valid_times[::step])
    plt.xticks(rotation=45)  # 标签旋转45°，避免重叠

    # 7. 调整布局并显示图表
    plt.tight_layout()
    plt.show()



In [ ]:
# ===================== 新增：实际墙面太阳辐射强度计算（W/m²）=====================
def get_day_of_year(dt: datetime) -> int:
    """计算一年中的第几天（积日n）"""
    return dt.timetuple().tm_yday


def clear_sky_DNI(dt: datetime) -> float:
    """
    标准晴空模型计算法向直射辐照度DNI (W/m²)
    适用：无实测气象数据时的理论计算
    """
    I_sc = 1367.0  # 太阳常数 W/m²
    n = get_day_of_year(dt)
    
    # 1. 日地距离修正（表观太阳常数）
    orbital_factor = 1.0 + 0.033 * math.cos(2 * math.pi * n / 365)
    I_app = I_sc * orbital_factor
    
    # 2. 计算该时刻太阳仰角（复用已有函数，仅用来获取高度角）
    # 这里需要经纬度，临时用一个占位，实际会在外部传入，此函数仅做DNI大气衰减
    # 真正仰角会在主函数中传入，避免重复计算太阳位置
    return I_app, orbital_factor


def calculate_actual_wall_radiation(
    date_time: datetime, 
    lat: float, 
    lng: float,
    custom_DNI: float = None  # 允许传入实测DNI，优先使用
) -> dict:
    """
    计算东/南/西墙面的【实际太阳辐射强度】，单位：W/m²
    :param date_time: 时刻
    :param lat: 纬度
    :param lng: 经度
    :param custom_DNI: 若有当地实测/气象DNI，传入覆盖晴空模型
    :return: 各墙面实际辐射(W/m²) + 中间物理量
    """
    # 1. 复用已有函数：获取墙面法向投影系数 I_norm
    wall_light = calculate_wall_sunlight_intensity(date_time, lat, lng)
    alt_deg = wall_light['altitude_deg']
    alt_rad = math.radians(alt_deg)
    
    # 太阳低于地平线，辐射为0
    if alt_deg <= 0:
        return {
            'datetime': date_time.strftime("%Y-%m-%d %H:%M"),
            'altitude_deg': alt_deg,
            'DNI': 0.0,
            'east_wall_radiation_Wm2': 0.0,
            'south_wall_radiation_Wm2': 0.0,
            'west_wall_radiation_Wm2': 0.0,
            'note': '太阳在地平线以下，无直射辐射'
        }
    
    # 2. 计算DNI（法向直射辐照度）
    I_app, orbital_factor = clear_sky_DNI(date_time)
    # ASHRAE晴空衰减项
    B = 0.23
    sin_h = math.sin(alt_rad)
    # 防止日出日落时分母过小
    sin_h = max(sin_h, 1e-6)
    
    DNI_model = I_app * math.exp(-B / sin_h)
    
    # 优先使用用户传入的实测DNI
    DNI = custom_DNI if custom_DNI is not None else DNI_model
    
    # 3. 计算墙面实际垂直辐射：I_wall = DNI * I_norm
    east_rad = DNI * wall_light['east_wall_intensity']
    south_rad = DNI * wall_light['south_wall_intensity']
    west_rad = DNI * wall_light['west_wall_intensity']
    
    return {
        'datetime': date_time.strftime("%Y-%m-%d %H:%M"),
        'altitude_deg': alt_deg,
        'orbital_factor': orbital_factor,
        'DNI_model_used': '实测DNI' if custom_DNI else '晴空模型DNI',
        'DNI_Wm2': round(DNI, 2),
        'east_wall_radiation_Wm2': round(east_rad, 2),
        'south_wall_radiation_Wm2': round(south_rad, 2),
        'west_wall_radiation_Wm2': round(west_rad, 2),
        'note': '太阳在地平线上，结果为墙面法向直射辐射(W/m²)'
    }

# ===================== 新增：可视化一天内东/南/西墙【实际辐射强度(W/m²)】趋势 =====================
def plot_daily_wall_radiation(target_date: datetime, lat: float, lng: float, time_interval_min=10):
    """
    绘制指定日期、地点，全天东/南/西墙的实际法向直射太阳辐射强度（单位：W/m²）
    :param target_date: 目标日期
    :param lat: 纬度
    :param lng: 经度
    :param time_interval_min: 采样时间间隔(分钟)
    """
    # 中文与绘图配置
    plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei']
    plt.rcParams['axes.unicode_minus'] = False
    plt.rcParams['figure.figsize'] = (12, 6)

    # 构造当天全时段采样点
    base_date = datetime(target_date.year, target_date.month, target_date.day)
    sample_times = [base_date + timedelta(minutes=i) for i in range(0, 24*60, time_interval_min)]

    time_labels = []
    east_rad = []
    south_rad = []
    west_rad = []

    for t in sample_times:
        rad = calculate_actual_wall_radiation(t, lat, lng)
        # 只保留太阳在地平线上的时刻
        if rad['altitude_deg'] > 0:
            time_labels.append(t.strftime("%H:%M"))
            east_rad.append(rad['east_wall_radiation_Wm2'])
            south_rad.append(rad['south_wall_radiation_Wm2'])
            west_rad.append(rad['west_wall_radiation_Wm2'])

    fig, ax = plt.subplots()
    # 三条墙面辐射曲线
    ax.plot(time_labels, east_rad, color='#1E90FF', linewidth=2.5, label='东墙实际辐射', marker='.', markersize=3)
    ax.plot(time_labels, south_rad, color='#FF4500', linewidth=2.5, label='南墙实际辐射', marker='.', markersize=3)
    ax.plot(time_labels, west_rad, color='#32CD32', linewidth=2.5, label='西墙实际辐射', marker='.', markersize=3)

    # 图表标注
    ax.set_title(f'东/南/西墙面实际太阳直射辐射强度全天变化\n'
                 f'日期：{base_date.strftime("%Y-%m-%d")} | 纬度：{lat:.1f}° 经度：{lng:.1f}°',
                 fontsize=14, pad=15)
    ax.set_xlabel('时刻', fontsize=12)
    ax.set_ylabel('实际法向直射辐射强度 / $W/m^2$', fontsize=12)
    ax.grid(True, linestyle='--', alpha=0.4)
    ax.legend(fontsize=11)
    ax.set_ylim(bottom=0)  # 辐射强度不为负

    # 控制x轴标签密度
    step = max(1, len(time_labels) // 12)
    ax.set_xticks(time_labels[::step])
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

In [ ]:
# ===================== 新增：采光专用 - 墙面直射照度计算(lux) =====================
def calculate_wall_direct_illuminance(date_time: datetime, lat: float, lng: float, custom_DNI=None):
    """
    建筑采光专用：计算东/南/西墙面的【直射自然光照度】
    单位：lux (勒克斯)，采光设计/照度模拟标准单位
    不含散射光，仅算直射阳光；与得热/气温无关，纯视觉采光量
    """
    # 1. 复用已有的实际辐射计算（拿到DNI和墙面投影系数）
    rad = calculate_actual_wall_radiation(date_time, lat, lng, custom_DNI=custom_DNI)
    
    # 2. 兼容太阳在地平线以下的情况（无DNI，照度为0）
    # 统一获取DNI，兼容两种返回结果
    dni = rad.get('DNI_Wm2', 0.0)  # 用get方法，不存在则返回0.0，避免KeyError
    if dni <= 1e-6:
        return {
            'datetime': date_time.strftime("%Y-%m-%d %H:%M"),
            'altitude_deg': rad.get('altitude_deg', 0.0),
            'DNI_Wm2': 0.0,
            'east_wall_direct_lux': 0,
            'south_wall_direct_lux': 0,
            'west_wall_direct_lux': 0,
            'unit': 'lux (勒克斯，采光直射照度)',
            'note': '太阳在地平线以下或无有效直射，照度为0'
        }
    
    # 3. 提取墙面投影系数（cosθ），直接从wall_light获取更准确，避免除法误差
    # 重新调用墙面光照系数函数，直接拿到纯净的cosθ
    wall_light = calculate_wall_sunlight_intensity(date_time, lat, lng)
    coef_east = wall_light['east_wall_intensity']
    coef_south = wall_light['south_wall_intensity']
    coef_west = wall_light['west_wall_intensity']

    # 4. 采光标准换算：DNI -> 可见光 -> 照度(lux)
    # 系数来源：可见光占比0.45，1 W/m²可见光 ≈ 120 lux
    lux_conversion = 0.45 * 120  # = 54
    
    east_lux = dni * coef_east * lux_conversion
    south_lux = dni * coef_south * lux_conversion
    west_lux = dni * coef_west * lux_conversion

    # 负照度无物理意义，截断为0（实际coef已经是max(0, ...)，这里做双重保障）
    east_lux = max(0.0, east_lux)
    south_lux = max(0.0, south_lux)
    west_lux = max(0.0, west_lux)

    return {
        'datetime': date_time.strftime("%Y-%m-%d %H:%M"),
        'altitude_deg': rad['altitude_deg'],
        'DNI_Wm2': round(dni, 2),
        'east_wall_direct_lux': round(east_lux),
        'south_wall_direct_lux': round(south_lux),
        'west_wall_direct_lux': round(west_lux),
        'unit': 'lux (勒克斯，采光直射照度)',
        'note': '仅直射阳光，无天空散射/地面反射，纯采光理论值'
    }

def plot_daily_wall_illuminance_lux(target_date: datetime, lat: float, lng: float, time_interval_min=10):
    plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei']
    plt.rcParams['axes.unicode_minus'] = False
    plt.rcParams['figure.figsize'] = (13, 6)

    # 1. 修正采样范围：从当地日出前1小时到日落后1小时，避免跨天错误
    base_date = datetime(target_date.year, target_date.month, target_date.day)
    # 替换原来的0-24*60，改为从4点到22点，覆盖绝大多数地区的日照时段
    sample_times = [base_date + timedelta(minutes=i) for i in range(4*60, 22*60, time_interval_min)]

    time_labels = []
    east_lux_list = []
    south_lux_list = []
    west_lux_list = []

    for t in sample_times:
        illu = calculate_wall_direct_illuminance(t, lat, lng)
        if illu['altitude_deg'] > 0:
            time_labels.append(t.strftime("%H:%M"))
            east_lux_list.append(illu['east_wall_direct_lux'])
            south_lux_list.append(illu['south_wall_direct_lux'])
            west_lux_list.append(illu['west_wall_direct_lux'])

    fig, ax = plt.subplots()
    ax.plot(time_labels, east_lux_list, color='#1E90FF', linewidth=2.2, label='东墙直射照度', markersize=2)
    ax.plot(time_labels, south_lux_list, color='#FF4500', linewidth=2.2, label='南墙直射照度', markersize=2)
    ax.plot(time_labels, west_lux_list, color='#32CD32', linewidth=2.2, label='西墙直射照度', markersize=2)

    ax.set_title(f'东/南/西墙面 直射自然光照度\n日期：{base_date.strftime("%Y-%m-%d")} | 纬度：{lat:.1f}° 经度：{lng:.1f}°',
                 fontsize=14, pad=16)
    ax.set_xlabel('时刻', fontsize=12)
    ax.set_ylabel('直射照度 / lux (勒克斯)', fontsize=12)
    ax.grid(True, linestyle='--', alpha=0.4)
    ax.legend(fontsize=11)
    ax.set_ylim(bottom=0)

    # 2. 修正横轴标签密度：根据有效时间长度自动调整，避免重叠
    step = max(1, len(time_labels) // 8)  # 减少标签数量，更清晰
    ax.set_xticks(time_labels[::step])
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

import math
from datetime import datetime, timedelta
import matplotlib.pyplot as plt

# 先确保你已有的 calculate_wall_direct_illuminance 和 calculate_wall_total_illuminance 函数已存在

# ===================== 新增：室内工作面照度计算（符合GB 50033，含指定参数） =====================
def calculate_indoor_work_plane_illuminance(date_time: datetime, lat: float, lng: float, custom_DNI=None):
    """
    计算室内工作面照度（距地面0.75m），基于指定参数：
    - 中空Low-E玻璃（τ=0.65）
    - 大型房间（会议室/教室，K=15）
    - 浅色调装修（ρ_avg=0.55）
    - 窗墙比：南墙0.45，东/西墙0.3
    最终结果可与GB 50033-2013对比达标情况
    """
    # 1. 固定参数取值（根据用户要求）
    tau = 0.65  # 中空Low-E玻璃透光系数
    K = 15.0    # 大型房间空间衰减系数
    rho_avg = 0.55  # 浅色调装修平均反射系数
    WWR = {
        'east': 0.3,   # 东墙窗墙比
        'south': 0.45, # 南墙窗墙比
        'west': 0.3    # 西墙窗墙比
    }

    # 2. 获取室外墙面总照度（直射+散射+地面反射）
    wall_total_illu = calculate_wall_total_illuminance(date_time, lat, lng, custom_DNI)
    alt_deg = wall_total_illu['altitude_deg']
    wall_east_total = wall_total_illu['east_total_lux']
    wall_south_total = wall_total_illu['south_total_lux']
    wall_west_total = wall_total_illu['west_total_lux']

    # 3. 步骤1：室外窗洞口照度 → 室内窗洞口内侧照度（窗户透光衰减）
    window_in_east = wall_east_total * tau
    window_in_south = wall_south_total * tau
    window_in_west = wall_west_total * tau

    # 4. 步骤2：室内窗洞口内侧 → 工作面照度（含窗墙比、空间衰减、反射）
    work_plane = {}
    work_plane['east'] = window_in_east * (WWR['east'] / K) * rho_avg
    work_plane['south'] = window_in_south * (WWR['south'] / K) * rho_avg
    work_plane['west'] = window_in_west * (WWR['west'] / K) * rho_avg

    # 5. 总工作面照度（东+南+西墙窗户共同贡献，取总和）
    work_plane['total'] = work_plane['east'] + work_plane['south'] + work_plane['west']

    # 6. GB 50033-2013 达标判断
    gb_standard = {
        'residential': 100,  # 住宅建筑最低照度
        'office_education': 300  # 办公/教育/医院建筑最低照度
    }
    judgment = {
        'residential': '达标' if work_plane['total'] >= gb_standard['residential'] else '不达标',
        'office_education': '达标' if work_plane['total'] >= gb_standard['office_education'] else '不达标'
    }

    # 7. 整理返回结果
    return {
        'datetime': date_time.strftime("%Y-%m-%d %H:%M"),
        'altitude_deg': alt_deg,
        # 室外墙面总照度
        'outdoor_east_total_lux': round(wall_east_total),
        'outdoor_south_total_lux': round(wall_south_total),
        'outdoor_west_total_lux': round(wall_west_total),
        # 室内工作面照度（分墙面贡献）
        'work_plane_east_lux': round(work_plane['east']),
        'work_plane_south_lux': round(work_plane['south']),
        'work_plane_west_lux': round(work_plane['west']),
        'work_plane_total_lux': round(work_plane['total']),
        # 达标判断
        'gb_residential_100lux': judgment['residential'],
        'gb_office_300lux': judgment['office_education'],
        'unit': 'lux（勒克斯），工作面距地面0.75m',
        'note': '参数：中空Low-E玻璃(τ=0.65)、大型房间(K=15)、浅色调装修(ρ=0.55)'
    }

# ===================== 新增：采光专用 - 墙面直射+散射+总光照度计算(lux)【符合GB 50033】 =====================
def calculate_wall_total_illuminance(date_time: datetime, lat: float, lng: float, custom_DNI=None):
    """
    建筑采光专用：计算东/南/西墙面的【直射+散射+总光照度】
    严格遵循《建筑采光设计标准 GB 50033-2013》和 CIE 全阴天天空模型
    """
    # 1. 复用已有函数，获取直射照度
    direct_illu = calculate_wall_direct_illuminance(date_time, lat, lng, custom_DNI)
    alt_deg = direct_illu['altitude_deg']
    alt_rad = math.radians(alt_deg)
    sin_h = math.sin(alt_rad) if alt_deg > 0 else 0.0
    sin_h = max(sin_h, 0.0)  # 避免夜间出现负散射

    # 2. 计算天空散射光照度（CIE 全阴天模型，GB 50033 推荐简化公式）
    # 水平面散射照度：E_h_diff = 10000 * sin_h
    # 垂直墙面散射照度：E_v_diff = 5000 * sin_h（GB 50033 推荐值）
    E_h_diff = 10000 * sin_h
    E_v_diff = 5000 * sin_h

    # 3. 计算地面反射光照度（可选，GB 50033 推荐简化）
    rho = 0.3  # 地面反射比（中等反射率，0.2~0.5 可调）
    F_wg = 0.2  # 墙面与地面角系数（工程简化值）
    E_reflected = rho * E_h_diff * F_wg

    # 4. 提取直射照度，计算总照度（直射+散射+地面反射）
    east_direct = direct_illu['east_wall_direct_lux']
    south_direct = direct_illu['south_wall_direct_lux']
    west_direct = direct_illu['west_wall_direct_lux']

    # 垂直墙面散射照度与朝向无关，三者散射值相同
    east_diff = E_v_diff
    south_diff = E_v_diff
    west_diff = E_v_diff

    # 总照度 = 直射 + 散射 + 地面反射
    east_total = east_direct + east_diff + E_reflected
    south_total = south_direct + south_diff + E_reflected
    west_total = west_direct + west_diff + E_reflected

    # 5. 整理返回结果
    return {
        'datetime': date_time.strftime("%Y-%m-%d %H:%M"),
        'altitude_deg': alt_deg,
        'sin_h': round(sin_h, 4),
        'wall_vertical_diffuse_lux': round(E_v_diff),
        'ground_reflected_lux': round(E_reflected),
        # 直射照度
        'east_direct_lux': round(east_direct),
        'south_direct_lux': round(south_direct),
        'west_direct_lux': round(west_direct),
        # 散射照度
        'east_diffuse_lux': round(east_diff),
        'south_diffuse_lux': round(south_diff),
        'west_diffuse_lux': round(west_diff),
        # 总照度
        'east_total_lux': round(east_total),
        'south_total_lux': round(south_total),
        'west_total_lux': round(west_total),
        'unit': 'lux (勒克斯)，采光设计标准单位',
        'note': '符合GB 50033-2013：直射+CIE全阴天散射+地面反射'
    }
    

    # ===================== 新增：室内工作面照度全天可视化（含达标阈值线） =====================
def plot_indoor_work_plane_illuminance(target_date: datetime, lat: float, lng: float, time_interval_min=10):
    """
    可视化室内工作面总照度全天变化，叠加GB 50033-2013达标阈值线
    """
    # 1. 绘图基础配置
    plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei']
    plt.rcParams['axes.unicode_minus'] = False
    plt.rcParams['figure.figsize'] = (15, 7)
    plt.rcParams['grid.alpha'] = 0.4

    # 2. 构造采样时间（4:00-22:00，覆盖日照时段）
    base_date = datetime(target_date.year, target_date.month, target_date.day)
    sample_times = [base_date + timedelta(minutes=i) for i in range(4*60, 22*60, time_interval_min)]

    # 3. 初始化数据存储列表
    time_labels = []
    work_plane_total = []
    work_plane_east = []
    work_plane_south = []
    work_plane_west = []

    # 4. 遍历采样时间，计算工作面照度
    for t in sample_times:
        work_illu = calculate_indoor_work_plane_illuminance(t, lat, lng)
        alt_deg = work_illu['altitude_deg']
        
        if alt_deg >= 0:  # 保留太阳高度角≥0的有效数据
            time_labels.append(t.strftime("%H:%M"))
            work_plane_total.append(work_illu['work_plane_total_lux'])
            work_plane_east.append(work_illu['work_plane_east_lux'])
            work_plane_south.append(work_illu['work_plane_south_lux'])
            work_plane_west.append(work_illu['work_plane_west_lux'])

    # 5. 绘制图表
    fig, ax = plt.subplots()

    # 绘制各墙面贡献的工作面照度
    ax.plot(time_labels, work_plane_east, color='#1E90FF', linewidth=2, label='东墙窗户贡献', alpha=0.7)
    ax.plot(time_labels, work_plane_south, color='#FF4500', linewidth=2, label='南墙窗户贡献', alpha=0.7)
    ax.plot(time_labels, work_plane_west, color='#32CD32', linewidth=2, label='西墙窗户贡献', alpha=0.7)
    # 绘制总工作面照度（加粗突出）
    ax.plot(time_labels, work_plane_total, color='#8B008B', linewidth=3, label='工作面总照度', alpha=0.9)

    # 叠加GB 50033-2013达标阈值线
    ax.axhline(y=100, color='#FFA500', linestyle='--', linewidth=2, label='住宅达标阈值（100 lux）')
    ax.axhline(y=300, color='#FF0000', linestyle='--', linewidth=2, label='办公/教室达标阈值（300 lux）')

    # 图表标注与优化
    ax.set_title(f'室内工作面总照度全天变化（大型房间+中空Low-E玻璃+浅色调装修）\n日期：{base_date.strftime("%Y-%m-%d")} | 纬度：{lat:.1f}° 经度：{lng:.1f}°',
                 fontsize=14, pad=16)
    ax.set_xlabel('时刻', fontsize=12)
    ax.set_ylabel('工作面照度 / lux（距地面0.75m）', fontsize=12)
    ax.grid(True, linestyle='--')
    ax.legend(fontsize=10)
    ax.set_ylim(bottom=0)

    # 优化横轴标签，避免重叠
    step = max(1, len(time_labels) // 12)
    ax.set_xticks(time_labels[::step])
    plt.xticks(rotation=45)

    # 调整布局
    plt.tight_layout()
    plt.show()

    # ===================== 新增：采光可视化 - 符合GB 50033 直射+散射+地面反射+总照度(lux) =====================
def plot_daily_wall_standard_total_illuminance_lux(target_date: datetime, lat: float, lng: float, time_interval_min=10):
    """
    符合《建筑采光设计标准 GB 50033-2013》的可视化：
    东/南/西墙 直射+散射+地面反射+总光照度 全天变化曲线
    清晰区分4个分量，更贴合工程实际采光场景
    """
    # 1. 绘图基础配置
    plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei']
    plt.rcParams['axes.unicode_minus'] = False
    plt.rcParams['figure.figsize'] = (16, 10)
    plt.rcParams['grid.alpha'] = 0.4

    # 2. 构造合理采样时间（4:00-22:00，覆盖日照时段，避免夜间干扰）
    base_date = datetime(target_date.year, target_date.month, target_date.day)
    sample_times = [base_date + timedelta(minutes=i) for i in range(4*60, 22*60, time_interval_min)]

    # 3. 初始化数据存储列表
    time_labels = []
    # 东墙数据
    east_direct = []
    east_diff = []
    east_reflect = []
    east_total = []
    # 南墙数据
    south_direct = []
    south_diff = []
    south_reflect = []
    south_total = []
    # 西墙数据
    west_direct = []
    west_diff = []
    west_reflect = []
    west_total = []

    # 4. 遍历采样时间，计算标准完整照度
    for t in sample_times:
        total_illu = calculate_wall_total_illuminance(t, lat, lng)
        alt_deg = total_illu['altitude_deg']
        
        # 保留有效数据（太阳高度角>=0，包含日出日落前后的散射/反射光）
        if alt_deg >= 0:
            time_labels.append(t.strftime("%H:%M"))
            # 东墙数据填充
            east_direct.append(total_illu['east_direct_lux'])
            east_diff.append(total_illu['east_diffuse_lux'])
            east_reflect.append(total_illu['ground_reflected_lux'])
            east_total.append(total_illu['east_total_lux'])
            # 南墙数据填充
            south_direct.append(total_illu['south_direct_lux'])
            south_diff.append(total_illu['south_diffuse_lux'])
            south_reflect.append(total_illu['ground_reflected_lux'])
            south_total.append(total_illu['south_total_lux'])
            # 西墙数据填充
            west_direct.append(total_illu['west_direct_lux'])
            west_diff.append(total_illu['west_diffuse_lux'])
            west_reflect.append(total_illu['ground_reflected_lux'])
            west_total.append(total_illu['west_total_lux'])

    # 5. 绘制子图（3个墙面，每个墙面4条曲线，清晰无重叠）
    fig, (ax1, ax2, ax3) = plt.subplots(3, 1, sharex=True, sharey=True)
    fig.suptitle(f'符合GB 50033-2013 东/南/西墙面 直射+散射+地面反射+总光照度\n日期：{base_date.strftime("%Y-%m-%d")} | 纬度：{lat:.1f}° 经度：{lng:.1f}°',
                 fontsize=15, y=0.98)

    # 5.1 东墙子图
    ax1.plot(time_labels, east_direct, color='#1E90FF', linewidth=2, label='直射照度', alpha=0.8, marker='.', markersize=1)
    ax1.plot(time_labels, east_diff, color='#6495ED', linewidth=2, label='天空散射照度', alpha=0.8, marker='.', markersize=1)
    ax1.plot(time_labels, east_reflect, color='#B0C4DE', linewidth=2, label='地面反射照度', alpha=0.8, marker='.', markersize=1)
    ax1.plot(time_labels, east_total, color='#0000CD', linewidth=3, label='总照度（直射+散射+反射）', alpha=0.9)
    ax1.set_title('东墙', fontsize=13)
    ax1.grid(True, linestyle='--')
    ax1.legend(fontsize=10, loc='upper left')
    ax1.set_ylabel('照度 / lux（勒克斯）', fontsize=11)

    # 5.2 南墙子图
    ax2.plot(time_labels, south_direct, color='#FF4500', linewidth=2, label='直射照度', alpha=0.8, marker='.', markersize=1)
    ax2.plot(time_labels, south_diff, color='#FF7F50', linewidth=2, label='天空散射照度', alpha=0.8, marker='.', markersize=1)
    ax2.plot(time_labels, south_reflect, color='#FFC0CB', linewidth=2, label='地面反射照度', alpha=0.8, marker='.', markersize=1)
    ax2.plot(time_labels, south_total, color='#DC143C', linewidth=3, label='总照度（直射+散射+反射）', alpha=0.9)
    ax2.set_title('南墙', fontsize=13)
    ax2.grid(True, linestyle='--')
    ax2.legend(fontsize=10, loc='upper left')
    ax2.set_ylabel('照度 / lux（勒克斯）', fontsize=11)

    # 5.3 西墙子图
    ax3.plot(time_labels, west_direct, color='#32CD32', linewidth=2, label='直射照度', alpha=0.8, marker='.', markersize=1)
    ax3.plot(time_labels, west_diff, color='#90EE90', linewidth=2, label='天空散射照度', alpha=0.8, marker='.', markersize=1)
    ax3.plot(time_labels, west_reflect, color='#98FB98', linewidth=2, label='地面反射照度', alpha=0.8, marker='.', markersize=1)
    ax3.plot(time_labels, west_total, color='#006400', linewidth=3, label='总照度（直射+散射+反射）', alpha=0.9)
    ax3.set_title('西墙', fontsize=13)
    ax3.grid(True, linestyle='--')
    ax3.legend(fontsize=10, loc='upper left')
    ax3.set_xlabel('时刻', fontsize=11)
    ax3.set_ylabel('照度 / lux（勒克斯）', fontsize=11)

    # 6. 优化横轴标签，避免重叠
    step = max(1, len(time_labels) // 12)
    ax3.set_xticks(time_labels[::step])
    plt.xticks(rotation=45)

    # 7. 调整布局，保证图表完整显示
    plt.tight_layout()
    plt.subplots_adjust(top=0.92)
    plt.show()

In [ ]:
# ===================== 新增：按你自定义公式的遮阳采光计算 =====================
import math
from datetime import datetime, timedelta
import matplotlib.pyplot as plt

def calculate_shaded_illuminance_custom(
    date_time: datetime,
    lat: float,
    lng: float,
    H=6.0,
    l=1.0,
    h0=1.0,
    theta_deg=30.0,
    wall='south'  # 'east'/'south'/'west'
):
    """
    完全按用户自定义公式计算【带遮阳的墙面总照度】
    lambda = (H - l*cos(phi - theta)/cos(phi)) / H
    E_total = lambda * L1 + L2
    L1: 直射照度
    L2: 散射+地面反射(与遮阳无关)
    phi: 太阳光线与墙面外法线夹角
    """
    # 1. 获取无遮挡的总照度分量（直射、散射、反射）
    total_illu = calculate_wall_total_illuminance(date_time, lat, lng)
    # 2. 获取墙面法向投影系数 cos(phi)
    wall_coef = calculate_wall_sunlight_intensity(date_time, lat, lng)
    
    # 3. 按墙面选取对应 L1, L2, cos_phi
    if wall == 'east':
        L1 = total_illu['east_direct_lux']
        L2 = total_illu['east_diffuse_lux'] + total_illu['ground_reflected_lux']
        cos_phi = wall_coef['east_wall_intensity']
    elif wall == 'west':
        L1 = total_illu['west_direct_lux']
        L2 = total_illu['west_diffuse_lux'] + total_illu['ground_reflected_lux']
        cos_phi = wall_coef['west_wall_intensity']
    elif wall == 'south':
        L1 = total_illu['south_direct_lux']
        L2 = total_illu['south_diffuse_lux'] + total_illu['ground_reflected_lux']
        cos_phi = wall_coef['south_wall_intensity']
    else:
        raise ValueError("wall must be 'east'/'south'/'west'")
    
    alt_deg = total_illu['altitude_deg']
    # 太阳在地平线以下，全部为0

    if alt_deg <= 0:
        return {
            'datetime': date_time.strftime("%Y-%m-%d %H:%M"),
            'alt_deg': round(alt_deg, 2),
            'phi_deg': 0.0,  # 补齐缺失的 phi_deg 键
            'cos_phi': 0.0,
            'lambda': 0.0,
            'L1_direct': 0.0,  # 与另一分支的 L1 键名统一（原分支是 L1_direct）
            'L2_diffuse_reflect': 0.0,  # 与另一分支的 L2 键名统一（原分支是 L2_diffuse_reflect）
            'E_shaded_total': 0.0,
            'params': f'H={H},l={l},h0={h0},theta={theta_deg}°',  # 补齐 params 键
            'wall': wall,  # 补齐 wall 键
            'note': 'sun below horizon'
        }
    
    theta_rad = math.radians(theta_deg)
    
    # 4. 计算 phi（太阳与墙面法线夹角）
    # cos_phi 已经是 [0,1]，直接反求 phi
    cos_phi = max(cos_phi, 1e-9)  # 防除0
    phi_rad = math.acos(cos_phi)
    
    # 5. 按你公式计算遮阳系数 lambda
    numerator_cos = math.cos(phi_rad - theta_rad)
    term = l * numerator_cos / cos_phi
    lambda_val = (H - term) / H
    
    # 物理截断：不能为负，完全遮挡则lambda=0
    lambda_val = max(lambda_val, 0.0)
    
    # 6. 有遮阳后的总照度
    E_shaded = lambda_val * L1 + L2
    
    return {
        'datetime': date_time.strftime("%Y-%m-%d %H:%M"),
        'alt_deg': round(alt_deg, 2),
        'phi_deg': round(math.degrees(phi_rad), 2),
        'cos_phi': round(cos_phi, 4),
        'lambda': round(lambda_val, 4),
        'L1_direct': round(L1),
        'L2_diffuse_reflect': round(L2),
        'E_shaded_total': round(E_shaded),
        'params': f'H={H},l={l},h0={h0},theta={theta_deg}°',
        'wall': wall
    }

# ===================== 新增：全天遮阳后总照度变化曲线绘图 =====================
def plot_daily_shaded_illuminance(
    target_date: datetime,
    lat: float,
    lng: float,
    H=6.0,
    l=1.0,
    h0=1.0,
    theta_deg=30.0,
    wall='south',
    time_interval_min=10
):
    plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei']
    plt.rcParams['axes.unicode_minus'] = False
    plt.rcParams['figure.figsize'] = (15, 7)

    base_date = datetime(target_date.year, target_date.month, target_date.day)
    sample_times = [base_date + timedelta(minutes=i) for i in range(4*60, 22*60, time_interval_min)]

    time_labels = []
    lambda_list = []
    L1_list = []
    L2_list = []
    E_shaded_list = []

    for t in sample_times:
        res = calculate_shaded_illuminance_custom(
            t, lat, lng, H, l, h0, theta_deg, wall
        )
        alt = res['alt_deg']
        if alt > 0:
            time_labels.append(t.strftime("%H:%M"))
            lambda_list.append(res['lambda'])
            L1_list.append(res['L1_direct'])
            L2_list.append(res['L2_diffuse_reflect'])
            E_shaded_list.append(res['E_shaded_total'])

    fig, (ax1, ax2) = plt.subplots(2, 1, sharex=True, figsize=(15,9))
    fig.suptitle(
        f'自定义遮阳模型 | {wall}墙 | H={H},l={l},θ={theta_deg}° | {base_date.strftime("%Y-%m-%d")} | {lat:.1f}°N',
        fontsize=14
    )

    # 上图：遮阳系数 λ 全天变化
    ax1.plot(time_labels, lambda_list, 'r-o', ms=2, linewidth=2, label=r'$\lambda$ 遮阳系数')
    ax1.set_ylabel(r'$\lambda$')
    ax1.set_ylim(0, 1.05)
    ax1.grid(True, linestyle='--')
    ax1.legend()
    ax1.set_title('遮阳系数 λ 日内变化')

    # 下图：有遮阳总照度 + 分量
    ax2.plot(time_labels, L1_list, 'orange', linewidth=2, label='L1 直射照度')
    ax2.plot(time_labels, L2_list, 'c', linewidth=2, label='L2 散射+反射')
    ax2.plot(time_labels, E_shaded_list, 'k', linewidth=3, label='E_shaded = λ·L1 + L2 总照度')
    ax2.set_xlabel('时刻')
    ax2.set_ylabel('照度 / lux')
    ax2.grid(True, linestyle='--')
    ax2.legend()
    ax2.set_ylim(bottom=0)

    step = max(1, len(time_labels)//12)
    ax2.set_xticks(time_labels[::step])
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.subplots_adjust(top=0.92)
    plt.show()

In [ ]:
# ===================== 新增：自定义遮阳模型 → 室内工作面照度 =====================
def calculate_shaded_indoor_work_plane(
    date_time: datetime,
    lat: float,
    lng: float,
    H=6.0,
    l=1.0,
    h0=1.0,
    theta_deg=30.0,
    wall='south'  # 'east'/'south'/'west'
):
    """
    完整链路：
    自定义外墙遮阳 → 遮阳后墙面照度 → 透过Low-E玻璃 → 大型房间室内工作面照度
    参数：
    - H=6, l=1, h0=1, theta=30°（你指定）
    - 中空Low-E玻璃 τ=0.65
    - 大型教室/会议室 K=15
    - 浅色调装修 ρ_avg=0.55
    - 窗墙比：南0.45，东西0.3
    """
    # 1. 计算【外墙被遮阳后的墙面照度】（完全用你的公式）
    shaded_wall = calculate_shaded_illuminance_custom(
        date_time, lat, lng, H, l, h0, theta_deg, wall
    )
    
    E_wall_shaded = shaded_wall['E_shaded_total']
    alt_deg = shaded_wall['alt_deg']
    phi_deg = shaded_wall['phi_deg']
    lambda_val = shaded_wall['lambda']
    
    # 太阳低于地平线，直接返回0
    if alt_deg <= 0:
        return {
            'datetime': date_time.strftime("%Y-%m-%d %H:%M"),
            'wall': wall,
            'alt_deg': alt_deg,
            'phi_deg': phi_deg,
            'shading_lambda': lambda_val,
            'E_wall_shaded': 0.0,
            'E_work_shaded': 0.0,
            'gb_res_100lux': '不达标',
            'gb_office_300lux': '不达标',
            'note': '太阳在地平线以下'
        }
    
    # 2. 固定室内换算参数（你指定的建筑条件）
    tau = 0.65        # 中空Low-E玻璃
    K = 15.0          # 大型房间(教室/会议室)
    rho_avg = 0.55    # 浅色调装修
    
    # 窗墙比
    if wall == 'east':
        WWR = 0.3
    elif wall == 'west':
        WWR = 0.3
    elif wall == 'south':
        WWR = 0.45
    else:
        WWR = 0.3
    
    # 3. 换算到室内工作面照度
    E_work_shaded = E_wall_shaded * tau * (WWR / K) * rho_avg
    
    # 4. 达标判断（GB 50033-2013）
    gb_res = 100
    gb_office_edu = 300
    judge_res = "达标" if E_work_shaded >= gb_res else "不达标"
    judge_office = "达标" if E_work_shaded >= gb_office_edu else "不达标"
    
    return {
        'datetime': date_time.strftime("%Y-%m-%d %H:%M"),
        'wall': wall,
        'params': f'H={H},l={l},theta={theta_deg}°',
        'alt_deg': round(alt_deg, 2),
        'phi_deg': round(phi_deg, 2),
        'shading_lambda': round(lambda_val, 4),
        'E_wall_shaded': round(E_wall_shaded),       # 遮阳后外墙照度
        'E_work_shaded': round(E_work_shaded),       # 最终室内工作面照度
        'gb_residential_100lux': judge_res,
        'gb_office_classroom_300lux': judge_office,
        'unit': 'lux'
    }

# ===================== 新增：全天【遮阳后室内工作面照度】可视化 =====================
def plot_daily_shaded_indoor_work_plane(
    target_date: datetime,
    lat: float,
    lng: float,
    H=6.0,
    l=1.0,
    theta_deg=30.0,
    wall='south',
    time_interval_min=10
):
    plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei']
    plt.rcParams['axes.unicode_minus'] = False
    
    base_date = datetime(target_date.year, target_date.month, target_date.day)
    sample_times = [base_date + timedelta(minutes=i) for i in range(4*60, 22*60, time_interval_min)]
    
    time_labels = []
    lambda_list = []
    E_wall_list = []
    E_work_list = []
    
    for t in sample_times:
        res = calculate_shaded_indoor_work_plane(
            t, lat, lng, H, l, 1.0, theta_deg, wall
        )
        if res['alt_deg'] > 0:
            time_labels.append(t.strftime("%H:%M"))
            lambda_list.append(res['shading_lambda'])
            E_wall_list.append(res['E_wall_shaded'])
            E_work_list.append(res['E_work_shaded'])
    
    fig, (ax2, ax3) = plt.subplots(2, 1, sharex=True, figsize=(16, 10))
    fig.suptitle(
        f'自定义遮阳 → 室内工作面照度 | {wall}墙 | H={H},l={l},θ={theta_deg}°\n'
        f'{base_date.strftime("%Y-%m-%d")} | 纬度{lat:.1f}°N | 教室/会议室+Low-E玻璃',
        fontsize=14
    )
    
    
    # 子图2：遮阳后外墙照度
    ax2.plot(time_labels, E_wall_list, 'orange', linewidth=2.5, label='外墙遮阳后照度')
    ax2.set_ylabel('外墙照度 / lux')
    ax2.grid(True, linestyle='--')
    ax2.legend()
    ax2.set_title('外墙被遮阳后的墙面照度')
    
    # 子图3：室内工作面照度 + 国标阈值线
    ax3.plot(time_labels, E_work_list, 'k-', linewidth=3, label='室内工作面照度(遮阳后)')
    ax3.axhline(y=100, color='g', linestyle='--', linewidth=2, label='住宅最低 100 lux')
    ax3.axhline(y=300, color='m', linestyle='--', linewidth=2, label='教室/办公最低 300 lux')
    ax3.set_xlabel('时刻')
    ax3.set_ylabel('室内照度 / lux')
    ax3.grid(True, linestyle='--')
    ax3.legend()
    ax3.set_ylim(bottom=0)
    
    # 横轴优化
    step = max(1, len(time_labels) // 12)
    ax3.set_xticks(time_labels[::step])
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.subplots_adjust(top=0.92)
    plt.show()

In [ ]:
# ===================== 修复版：东/南/西三面墙 联合可视化（遮阳→室内工作面照度） =====================
def plot_3walls_joint_shaded_indoor_work_plane(
    target_date: datetime,
    lat: float,
    lng: float,
    H=6.0,
    l=1.0,
    theta_deg=30.0,
    time_interval_min=10
):
    # 绘图基础配置（修正后的正确格式）
    plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei']
    plt.rcParams['axes.unicode_minus'] = False
    
    # 定义要对比的三面墙
    walls = ['east', 'south', 'west']
    wall_colors = {'east': '#1E90FF', 'south': '#FF4500', 'west': '#32CD32'}
    wall_names = {'east': '东墙', 'south': '南墙', 'west': '西墙'}
    
    # 构造采样时间（覆盖完整24小时，适配赤道地区）
    base_date = datetime(target_date.year, target_date.month, target_date.day)
    sample_times = [base_date + timedelta(minutes=i) for i in range(0, 24*60, time_interval_min)]
    
    # 初始化数据存储（按墙面分类）
    joint_data = {}
    for wall in walls:
        joint_data[wall] = {
            'time_labels': [],
            'lambda_list': [],
            'E_wall_list': [],
            'E_work_list': []
        }
    
    # 遍历采样时间，计算三面墙的数据
    for t in sample_times:
        for wall in walls:
            res = calculate_shaded_indoor_work_plane(
                t, lat, lng, H, l, 1.0, theta_deg, wall
            )
            if res['alt_deg'] > 0:  # 仅保留太阳在地平线以上的数据
                joint_data[wall]['time_labels'].append(t.strftime("%H:%M"))
                joint_data[wall]['lambda_list'].append(res['shading_lambda'])
                joint_data[wall]['E_wall_list'].append(res['E_wall_shaded'])
                joint_data[wall]['E_work_list'].append(res['E_work_shaded'])
    
    # 绘制联合图表（3行1列：遮阳系数 → 外墙照度 → 室内工作面照度）
    fig, (ax2, ax3) = plt.subplots(2, 1, sharex=True, figsize=(18, 12))
    fig.suptitle(
        f'东/南/西三面墙 联合对比 | H={H},l={l},θ={theta_deg}°\n'
        f'{base_date.strftime("%Y-%m-%d")} | 纬度{lat:.1f}°N | 教室/会议室+Low-E玻璃',
        fontsize=15
    )
    
    
    # 子图2：三面墙 遮阳后外墙照度 对比（修复：自动适配纵轴范围）
    ax2.set_title('遮阳后外墙照度 日内变化（三面墙对比）')
    ax2.set_ylabel('外墙照度 / lux')
    ax2.grid(True, linestyle='--', alpha=0.6)
    # 自动适配纵轴，不再固定0-1
    all_E_wall = []
    for wall in walls:
        all_E_wall.extend(joint_data[wall]['E_wall_list'])
    if all_E_wall:
        ax2.set_ylim(bottom=0, top=max(all_E_wall) * 1.1)
    for wall in walls:
        ax2.plot(
            joint_data[wall]['time_labels'],
            joint_data[wall]['E_wall_list'],
            color=wall_colors[wall],
            linewidth=2.5,
            label=f'{wall_names[wall]}'
        )
    ax2.legend(fontsize=10)
    
    # 子图3：三面墙 室内工作面照度 对比（叠加国标阈值）
    ax3.set_title('室内工作面照度 日内变化（三面墙对比）')
    ax3.set_xlabel('时刻')
    ax3.set_ylabel('室内工作面照度 / lux')
    ax3.grid(True, linestyle='--', alpha=0.6)
    # 自动适配纵轴，不再固定0-1
    all_E_work = []
    for wall in walls:
        all_E_work.extend(joint_data[wall]['E_work_list'])
    if all_E_work:
        ax3.set_ylim(bottom=0, top=max(all_E_work) * 1.1)
    
    # 绘制三面墙数据
    for wall in walls:
        ax3.plot(
            joint_data[wall]['time_labels'],
            joint_data[wall]['E_work_list'],
            color=wall_colors[wall],
            linewidth=2.5,
            label=f'{wall_names[wall]}'
        )
    
    # 叠加国标阈值线（加粗，方便对比）
    ax3.axhline(y=100, color='g', linestyle='--', linewidth=2, label='住宅最低阈值 100 lux')
    ax3.axhline(y=300, color='m', linestyle='--', linewidth=2, label='教室/办公最低阈值 300 lux')
    ax3.legend(fontsize=10)
    
    # 优化横轴标签（用南墙时间标签作为基准，确保完整）
    if joint_data['south']['time_labels']:
        first_wall_time = joint_data['south']['time_labels']
        step = max(1, len(first_wall_time) // 12)
        ax3.set_xticks(first_wall_time[::step])
    plt.xticks(rotation=45)
    
    # 调整布局，避免标题/标签重叠
    plt.tight_layout()
    plt.subplots_adjust(top=0.90)
    plt.show()

In [ ]:
# ===================== 辅助函数：计算无遮挡时的室内工作面照度（无遮阳） =====================
def calculate_unshaded_indoor_work_plane(
    date_time: datetime,
    lat: float,
    lng: float,
    wall='south'
):
    """
    计算无遮阳时的室内工作面照度（作为对照基准）
    参数：与带遮阳函数一致，无l和theta（无遮阳）
    建筑参数：固定（中空Low-E玻璃、大型教室、浅色调装修、对应窗墙比）
    """
    # 固定建筑参数（与带遮阳函数保持一致，保证对照的公平性）
    tau = 0.65        # 中空Low-E玻璃透光系数
    K = 15.0          # 大型教室/会议室空间衰减系数
    rho_avg = 0.55    # 浅色调装修平均反射系数
    WWR = {
        'east': 0.3,
        'south': 0.45,
        'west': 0.3
    }
    
    # 1. 获取无遮挡的外墙总照度（直射+散射+反射）
    total_illu = calculate_wall_total_illuminance(date_time, lat, lng)
    alt_deg = total_illu['altitude_deg']
    
    # 太阳在地平线以下，返回0
    if alt_deg <= 0:
        return 0.0
    
    # 2. 按墙面选取无遮挡外墙总照度
    if wall == 'east':
        E_wall_unshaded = total_illu['east_total_lux']
    elif wall == 'west':
        E_wall_unshaded = total_illu['west_total_lux']
    elif wall == 'south':
        E_wall_unshaded = total_illu['south_total_lux']
    else:
        raise ValueError("wall must be 'east'/'south'/'west'")
    
    # 3. 换算为无遮挡室内工作面照度
    E_work_unshaded = E_wall_unshaded * tau * (WWR[wall] / K) * rho_avg
    
    # 边界处理：无负值
    return max(E_work_unshaded, 0.0)

# ===================== 核心函数：计算单个采样点的优化目标量 J =====================
def calculate_single_J(
    date_time: datetime,
    lat: float,
    lng: float,
    l=1.0,
    theta_deg=30.0,
    wall='south'
):
    """
    计算单个时刻的优化目标量 J
    J = 1/6*((当前室内光强-300)/300) + 5/6*(当前室内光强/无遮挡室内光强)
    """
    # 1. 获取带遮阳的当前室内光强（当前光强）
    shaded_res = calculate_shaded_indoor_work_plane(
        date_time, lat, lng, H=6.0, l=l, theta_deg=theta_deg, wall=wall
    )
    current_indoor = max(shaded_res['E_work_shaded'], 0.0)  # 边界处理：无负值
    
    # 2. 获取无遮挡的室内光强（对照基准）
    unshaded_indoor = calculate_unshaded_indoor_work_plane(
        date_time, lat, lng, wall=wall
    )
    
    # 3. 计算 J 的两部分
    # 第一部分：与300lux的偏差率
    part1 = (current_indoor - 300.0) / 300.0
    
    # 第二部分：遮阳光强保留率（避免除0错误）
    if unshaded_indoor <= 1e-9:  # 无遮挡光强接近0，直接取0
        part2 = 0.0
    else:
        part2 = current_indoor / unshaded_indoor
    
    # 4. 合并计算 J
    J = (1/6) * part1 + (5/6) * part2
    
    return {
        'datetime': date_time.strftime("%Y-%m-%d %H:%M"),
        'current_indoor_lux': round(current_indoor, 2),
        'unshaded_indoor_lux': round(unshaded_indoor, 2),
        'part1': round(part1, 4),
        'part2': round(part2, 4),
        'J': round(J, 4),
        'wall': wall,
        'params': f'l={l}m, theta={theta_deg}°'
    }

# ===================== 核心函数：计算当日优化目标量 J_day（按时段统计） =====================
def calculate_daily_J(
    target_date: datetime,
    lat: float,
    lng: float,
    l=1.0,
    theta_deg=30.0,
    wall='south',
    time_interval_min=10
):
    """
    计算当日的优化目标量 J_day：
    1. 划分8:00-13:00、13:00-18:00两个时段
    2. 分别计算两个时段J的平均值
    3. 取两个平均值的最大值作为当日J
    """
    # 1. 基础配置：时段划分、采样时间构造
    base_date = datetime(target_date.year, target_date.month, target_date.day)
    # 定义两个时段（开始时间，结束时间）
    time_periods = [
        (base_date.replace(hour=8, minute=0), base_date.replace(hour=13, minute=0)),  # 8-13点
        (base_date.replace(hour=13, minute=0), base_date.replace(hour=18, minute=0))   # 13-18点
    ]
    J_period_avg = []  # 存储两个时段的J平均值
    
    # 2. 遍历每个时段，计算J平均值
    for period_start, period_end in time_periods:
        # 构造该时段内的采样时间
        period_minutes = int((period_end - period_start).total_seconds() / 60)
        sample_times = [
            period_start + timedelta(minutes=i)
            for i in range(0, period_minutes + 1, time_interval_min)
        ]
        
        # 存储该时段内所有有效J值
        period_J_list = []
        for t in sample_times:
            # 计算单个点J
            single_J_res = calculate_single_J(t, lat, lng, l, theta_deg, wall)
            J_value = single_J_res['J']
            
            # 仅保留太阳在地平线以上的有效数据（避免夜间无效值干扰）
            shaded_res = calculate_shaded_indoor_work_plane(t, lat, lng, l=l, theta_deg=theta_deg, wall=wall)
            if shaded_res['alt_deg'] > 0:
                period_J_list.append(J_value)
        
        # 计算该时段J的平均值（无有效数据时取0）
        if len(period_J_list) > 0:
            period_J_avg = sum(period_J_list) / len(period_J_list)
        else:
            period_J_avg = 0.0
        
        J_period_avg.append(round(period_J_avg, 4))
    
    # 3. 取两个时段平均值的最大值作为当日J
    J_day = max(J_period_avg)
    
    # 4. 整理返回结果
    return {
        'date': base_date.strftime("%Y-%m-%d"),
        'wall': wall,
        'params': f'l={l}m, theta={theta_deg}°',
        'time_interval_min': time_interval_min,
        'period1_8_13_avg_J': J_period_avg[0],  # 8-13点平均值
        'period2_13_18_avg_J': J_period_avg[1], # 13-18点平均值
        'J_day': round(J_day, 4),  # 当日最终J（取两个时段最大值）
        'note': 'J_day为8-13点和13-18点J平均值的最大值，目标向0接近'
    }

In [ ]:
# 导入必要库
from datetime import datetime, timedelta


global TARGET_LAT, TARGET_LNG, L_VALUE, THETA_VALUE, WALL
TARGET_DATE=datetime(2025,12,22)
TARGET_LAT = 40
TARGET_LNG = 0
L_VALUE = 1.0
THETA_VALUE = 60.0
WALL = 'south'

# 对比1：θ=30°（原参数）
plot_3walls_joint_shaded_indoor_work_plane(
    target_date=TARGET_DATE,
    lat=TARGET_LAT,
    lng=TARGET_LNG,
    H=6.0,
    l=L_VALUE,
    theta_deg=THETA_VALUE,
    time_interval_min=10
)


# 1. 测试单个采样点J
test_time = datetime(2025, 12, 22, 12, 0)
single_J_res = calculate_single_J(
    test_time, TARGET_LAT, TARGET_LNG,
    l=L_VALUE, theta_deg=THETA_VALUE, wall=WALL
)
print("=== 单个采样点J计算结果 ===")
for k, v in single_J_res.items():
    print(f"{k}: {v}")

# 2. 计算当日J_day
daily_J_res = calculate_daily_J(
    TARGET_DATE, TARGET_LAT, TARGET_LNG,
    l=L_VALUE, theta_deg=THETA_VALUE, wall=WALL,
    time_interval_min=10
)
print("\n=== 当日J_day计算结果 ===")
for k, v in daily_J_res.items():
    print(f"{k}: {v}")